# Capacitated Vehicle Routing Problem with Time Windows (CVRPTW)
## Pharmaceutical Last-Mile Delivery — Stochastic Optimization Analysis

---

**Dataset:** Real-world pharmaceutical distribution network (Zenodo record 15672291)  
**Solver:** Google OR-Tools CP routing (guided local search)  
**Focus:** Stochastic travel times · Robustness under uncertainty · Scenario analysis

---

## 1. Problem Definition

A pharmaceutical distributor operates a fleet of refrigerated vans from a central depot to **78 hospital and pharmacy nodes** in the Attica region (Greece). Each customer has:

- A **weight** and **volume** demand
- A **service time** (unloading duration)
- A **time window** `[EAT, LAT]` — the earliest and latest acceptable arrival

Travel time is **stochastic**, characterised by three scenarios drawn from GPS trajectory data:

| Scenario | Interpretation |
|---|---|
| Optimistic | Free-flow traffic |
| Most-likely | Typical weekday conditions |
| Pessimistic | Peak-hour congestion |

**Decision:** Assign customers to vehicles and sequence deliveries to minimise total travel time while satisfying all capacity and time-window constraints.

## 2. Mathematical Formulation

### Sets and parameters

$$N = \{0, 1, \ldots, n\}, \quad 0 = \text{depot},\quad K = \{1,\ldots,m\} = \text{vehicles}$$

$$\tau_{ij} = \text{travel time on arc } (i,j),\quad w_i, v_i = \text{weight, volume demand at node } i$$

$$s_i = \text{service time},\quad [a_i, l_i] = \text{time window},\quad W, V = \text{vehicle capacities}$$

### Decision variables

$$x_{ijk} \in \{0,1\} \quad \text{vehicle } k \text{ traverses arc } (i,j)$$

$$t_i \geq 0 \quad \text{arrival time at node } i$$

### Objective

$$\min \sum_{k \in K} \sum_{(i,j) \in A} \tau_{ij} \cdot x_{ijk}$$

### Constraints

**Visit each customer exactly once:**
$$\sum_{k \in K} \sum_{j \in N} x_{ijk} = 1 \qquad \forall\, i \in N \setminus \{0\}$$

**Flow conservation:**
$$\sum_{j \in N} x_{ijk} = \sum_{j \in N} x_{jik} \qquad \forall\, i \in N,\; k \in K$$

**Weight and volume capacity:**
$$\sum_{i \in N} w_i \sum_{j \in N} x_{ijk} \leq W \qquad \forall\, k \in K$$
$$\sum_{i \in N} v_i \sum_{j \in N} x_{ijk} \leq V \qquad \forall\, k \in K$$

**Time propagation (with service time):**
$$t_j \geq t_i + s_i + \tau_{ij} - M(1 - x_{ijk}) \qquad \forall\, (i,j),\; k \in K$$

**Time window:**
$$a_i \leq t_i \leq l_i \qquad \forall\, i \in N$$

### Stochastic extension — PERT model

Travel time $\tilde{\tau}_{ij}$ is modelled as a PERT (four-parameter Beta) random variable:

$$\mu_{ij} = \frac{a_{ij} + 4m_{ij} + b_{ij}}{6}, \qquad \sigma^2_{ij} = \left(\frac{b_{ij} - a_{ij}}{6}\right)^2$$

where $a, m, b$ are the optimistic, most-likely, and pessimistic travel times respectively.  
Arrival-time variance is propagated additively along a route:

$$\sigma^2_{\text{arrival at node } p} = \sum_{\text{arcs on route up to } p} \sigma^2_{ij}$$

The probability of late delivery at node $i$ is then:

$$P(t_i > l_i) = 1 - \Phi\!\left(\frac{l_i - \mu_i^{\text{arrival}}}{\sigma_i^{\text{arrival}}}\right)$$

## 3. Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Project modules
from src.data_loader     import load_orders, load_day, orders_eda
from src.cvrptw_solver   import solve, CVRPTWSolver, DEFAULT_NUM_VEHICLES, DEFAULT_MAX_WEIGHT_KG, DEFAULT_MAX_VOLUME_M3
from src.stochastic_analysis  import (
    solve_all_scenarios, scenario_kpi_table, route_overlap_matrix,
    lateness_by_customer, compute_pert_statistics
)
from src.robustness_analysis  import (
    customer_delay_risk, route_stability_scores,
    arc_uncertainty_report, route_arrival_deviation
)
from src.scenario_experiments import run_all_experiments
from src.clustering           import (
    extract_node_features, choose_k, cluster_customers,
    solve_decomposed, compare_decomposed_vs_global
)
from src.visualizations import (
    plot_routes, plot_scenario_kpis, plot_arrival_vs_window,
    plot_delay_risk, plot_stability_scores, plot_experiment,
    plot_silhouette, plot_cluster_assignment, plot_kpi_dashboard,
    plot_time_matrix_heatmap, plot_pert_distribution
)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'sans-serif'})

OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Environment ready.')

## 4. Data Loading & Exploratory Analysis

In [ ]:
# Load customer orders (shared across all days)
orders = load_orders()
print('=== Orders Dataset ===')
orders_eda(orders)

In [ ]:
orders.describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Weight distribution
axes[0].hist(orders['WEIGHT'], bins=20, color='#2196F3', edgecolor='white', linewidth=0.8)
axes[0].set_xlabel('Weight (kg)');  axes[0].set_ylabel('Count')
axes[0].set_title('Demand Weight Distribution')
axes[0].spines[['top','right']].set_visible(False)

# Volume distribution
axes[1].hist(orders['VOLUME'], bins=20, color='#4CAF50', edgecolor='white', linewidth=0.8)
axes[1].set_xlabel('Volume (m³)');  axes[1].set_ylabel('Count')
axes[1].set_title('Demand Volume Distribution')
axes[1].spines[['top','right']].set_visible(False)

# Time window distribution
tw_counts = orders['LAT'].value_counts().sort_index()
axes[2].bar(tw_counts.index.astype(str), tw_counts.values, color='#FF9800', edgecolor='white')
axes[2].set_xlabel('LAT (min)');  axes[2].set_ylabel('Count')
axes[2].set_title('Time Window Distribution by LAT')
axes[2].spines[['top','right']].set_visible(False)

fig.suptitle('Customer Demand Profile', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/demand_profile.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Load Day 1 as the primary instance
data = load_day(1, orders)
print(f'Day 1: {data.num_customers} customers, {data.num_nodes} nodes (incl. depot)')
print()
print('Travel-time matrix statistics:')
print(data.summary())

In [ ]:
# Compare optimistic vs pessimistic travel-time matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scen, cmap in zip(axes, ['optimistic', 'pessimistic'], ['Blues', 'Reds']):
    tm = data.time_matrices[scen][:40, :40]
    im = ax.imshow(tm, cmap=cmap, aspect='auto')
    plt.colorbar(im, ax=ax, label='Travel time (min)')
    ax.set_title(f'{scen.capitalize()} Travel Times (first 40 nodes)', fontsize=11)
    ax.set_xlabel('To node');  ax.set_ylabel('From node')

fig.suptitle('Travel-Time Matrix Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/time_matrix_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PERT statistics across the arc set
mu_mat, var_mat = compute_pert_statistics(data)
a_mat = data.time_matrices['optimistic']
b_mat = data.time_matrices['pessimistic']

# Only non-zero arcs
mask  = a_mat > 0
ranges = (b_mat - a_mat)[mask]

print(f'Arc range (b-a) — mean: {ranges.mean():.1f} min  max: {ranges.max():.0f} min')
print(f'PERT σ per arc  — mean: {np.sqrt(var_mat[mask]).mean():.2f} min')

# PERT distribution for a high-uncertainty arc
ratio_mat = (b_mat - a_mat) / np.where(a_mat > 0, a_mat, 1)
i_max, j_max = np.unravel_index(ratio_mat.argmax(), ratio_mat.shape)
print(f'\nHighest uncertainty arc: {i_max} → {j_max}  '
      f'(opt={a_mat[i_max,j_max]:.0f}, ml={data.time_matrices["mostlikely"][i_max,j_max]:.0f}, '
      f'pess={b_mat[i_max,j_max]:.0f})')

In [ ]:
# Plot PERT distribution for the highest-uncertainty arc
fig = plot_pert_distribution(data, int(i_max), int(j_max),
                             save_path=f'{OUTPUT_DIR}/pert_arc_distribution.png')
plt.show()

## 5. Base CVRPTW Model — Most-Likely Scenario

We solve the CVRPTW under the most-likely travel times using OR-Tools' routing engine with **Guided Local Search** (GLS) as the metaheuristic. GLS escapes local optima by penalising recently used arc features.

**Fleet configuration:**
- Vehicles: 10
- Weight capacity: 500 kg / vehicle
- Volume capacity: 2.0 m³ / vehicle
- Time horizon: 480 minutes

In [ ]:
print('Solving base CVRPTW (most-likely scenario, Day 1) …')
result_ml = solve(
    data,
    scenario='mostlikely',
    num_vehicles=DEFAULT_NUM_VEHICLES,
    max_weight_kg=DEFAULT_MAX_WEIGHT_KG,
    max_volume_m3=DEFAULT_MAX_VOLUME_M3,
    time_limit_s=60,
)
print(result_ml)

In [ ]:
if result_ml.is_solved():
    fig = plot_routes(
        result_ml, data,
        title=f'Day 1 — Most-Likely Routes ({result_ml.num_vehicles_used} vehicles)',
        save_path=f'{OUTPUT_DIR}/routes_mostlikely.png'
    )
    plt.show()
else:
    print(f'Solver returned: {result_ml.status}')

In [ ]:
if result_ml.is_solved():
    fig = plot_arrival_vs_window(
        result_ml, data, max_routes=4,
        save_path=f'{OUTPUT_DIR}/arrival_windows_mostlikely.png'
    )
    plt.show()

## 6. Stochastic Scenario Analysis

We solve the same CVRPTW instance independently under all three travel-time scenarios.  
This mimics a **wait-and-see** strategy: the operator knows the day's traffic regime in advance.

The comparison reveals:
- How much the optimal routing cost varies under traffic conditions
- Which scenario produces the most late deliveries
- Whether the route topology changes significantly between scenarios

In [ ]:
print('Solving all three scenarios …')
scenario_results = solve_all_scenarios(data, time_limit_s=60)

In [ ]:
kpi_df = scenario_kpi_table(scenario_results)
print('=== KPI Comparison ===')
kpi_df

In [ ]:
fig = plot_scenario_kpis(kpi_df, save_path=f'{OUTPUT_DIR}/scenario_kpis.png')
plt.show()

In [ ]:
fig = plot_kpi_dashboard(scenario_results, save_path=f'{OUTPUT_DIR}/kpi_dashboard.png')
plt.show()

In [ ]:
overlap = route_overlap_matrix(scenario_results)
print('=== Route Sequence Overlap Matrix ===')
print('(Fraction of consecutive customer-pairs shared between scenarios)')
print(overlap.round(3))

In [ ]:
lateness_df = lateness_by_customer(data, scenario_results)
high_lateness = lateness_df[lateness_df['max_lateness'] > 0].sort_values('max_lateness', ascending=False)
print(f'Customers with lateness > 0 in any scenario: {len(high_lateness)}')
high_lateness.head(15)

## 7. Robustness Analysis

We fix the **most-likely routes** and evaluate how arrival times shift when actual traffic matches the optimistic or pessimistic scenario.  
This is the classical **a priori** robustness evaluation: build routes for nominal conditions, then stress-test them.

Two metrics are used:

1. **Arrival spread** — `pessimistic arrival − optimistic arrival` per customer; measures raw timing volatility.
2. **P(late)** — probability of exceeding the LAT, derived from the PERT variance propagated along the route.

In [ ]:
if result_ml.is_solved():
    # Re-simulate arrivals across scenarios on the fixed most-likely routes
    spread_df = route_arrival_deviation(result_ml, data)
    print(f'Top 10 customers by arrival spread:')
    print(spread_df[['route','node','optimistic','mostlikely','pessimistic','arrival_spread','lat']].head(10).to_string(index=False))

In [ ]:
if result_ml.is_solved():
    risk_df = customer_delay_risk(result_ml, data, late_prob_threshold=0.10)
    high_risk = risk_df[risk_df['high_risk']]
    print(f'High-risk customers (P(late) > 10%): {len(high_risk)}')
    print(risk_df[['node','mu_arrival','sigma_arrival','lat','slack_min','p_late','high_risk']]
          .head(15).to_string(index=False))

In [ ]:
if result_ml.is_solved() and not risk_df.empty:
    fig = plot_delay_risk(risk_df, threshold=0.10,
                          save_path=f'{OUTPUT_DIR}/delay_risk.png')
    plt.show()

In [ ]:
if result_ml.is_solved():
    stability_df = route_stability_scores(result_ml, data)
    print('Route stability scores:')
    print(stability_df.to_string(index=False))

In [ ]:
if result_ml.is_solved() and not stability_df.empty:
    fig = plot_stability_scores(stability_df, save_path=f'{OUTPUT_DIR}/route_stability.png')
    plt.show()

In [ ]:
if result_ml.is_solved():
    arc_df = arc_uncertainty_report(result_ml, data, top_n=15)
    print('Top 15 arcs by uncertainty ratio:')
    print(arc_df[['route','from','to','t_optimistic','t_mostlikely','t_pessimistic','uncertainty_ratio']]
          .to_string(index=False))

## 8. Scenario Experiments

We stress-test the solution structure across four operational dimensions:

| Experiment | Lever | Question |
|---|---|---|
| Demand scaling | Weight × volume × factor | At what load does feasibility break? |
| TW tightening | LAT × fraction | How sensitive is cost to SLA tightness? |
| Fleet reduction | Remove vehicles | What is the minimum viable fleet? |
| Capacity sensitivity | Reduce weight cap | Where is the capacity binding constraint? |

In [ ]:
exp_results = run_all_experiments(data, scenario='mostlikely', time_limit_s=40)

In [ ]:
fig = plot_experiment(
    exp_results['demand_scaling'],
    x_col='demand_scale', x_label='Demand Scale Factor',
    y_cols=['total_travel_min', 'late_deliveries', 'vehicles_used'],
    title='Experiment 1: Demand Scaling Impact',
    save_path=f'{OUTPUT_DIR}/exp_demand_scaling.png'
)
plt.show()
exp_results['demand_scaling']

In [ ]:
fig = plot_experiment(
    exp_results['tw_tightening'],
    x_col='lat_fraction', x_label='LAT Fraction (1.0 = original)',
    y_cols=['total_travel_min', 'late_deliveries'],
    title='Experiment 2: Time-Window Tightening Impact',
    save_path=f'{OUTPUT_DIR}/exp_tw_tightening.png'
)
plt.show()
exp_results['tw_tightening']

In [ ]:
fig = plot_experiment(
    exp_results['fleet_reduction'],
    x_col='num_vehicles_available', x_label='Vehicles Available',
    y_cols=['total_travel_min', 'late_deliveries'],
    title='Experiment 3: Fleet Size Reduction Impact',
    save_path=f'{OUTPUT_DIR}/exp_fleet_reduction.png'
)
plt.show()
exp_results['fleet_reduction']

In [ ]:
fig = plot_experiment(
    exp_results['capacity_sensitivity'],
    x_col='max_weight_kg', x_label='Max Weight Capacity (kg)',
    y_cols=['total_travel_min', 'vehicles_used'],
    title='Experiment 4: Weight Capacity Sensitivity',
    save_path=f'{OUTPUT_DIR}/exp_capacity_sensitivity.png'
)
plt.show()
exp_results['capacity_sensitivity']

## 9. Advanced OR: Clustering-Based Decomposition

With 78 customers, solving the monolithic CVRPTW requires a metaheuristic.  An alternative — and practically useful — approach is **geographic-temporal decomposition via k-means**:

1. Represent each customer in a feature space: `[from_depot_time, to_depot_time, mean_peer_time, EAT, LAT]`
2. Apply k-means to partition into $k$ clusters
3. Solve an independent (smaller) CVRPTW per cluster
4. Measure the **optimality gap** vs the global solve

The decomposition trades global optimality for faster, parallelisable computation — valuable in operational planning.

In [ ]:
X = extract_node_features(data)
best_k, sil_df = choose_k(X, k_range=range(2, 9))
print(f'Best k by silhouette: {best_k}')
print(sil_df.to_string(index=False))

In [ ]:
fig = plot_silhouette(sil_df, best_k, save_path=f'{OUTPUT_DIR}/silhouette_scores.png')
plt.show()

In [ ]:
cluster_df = cluster_customers(data, k=best_k)
print('Cluster assignment summary:')
print(cluster_df.groupby('cluster').agg(
    n_customers=('node_id','count'),
    total_weight=('weight','sum'),
    total_volume=('volume','sum')
).round(3))

In [ ]:
fig = plot_cluster_assignment(data, cluster_df, save_path=f'{OUTPUT_DIR}/cluster_assignment.png')
plt.show()

In [ ]:
print(f'Solving decomposed CVRPTW (k={best_k} clusters) …')
cluster_results, cluster_summary = solve_decomposed(
    data, k=best_k, scenario='mostlikely',
    vehicles_per_cluster=3, time_limit_s=30
)
print('\nCluster-level results:')
print(cluster_summary.to_string(index=False))

In [ ]:
comparison = compare_decomposed_vs_global(cluster_results, result_ml)
print('=== Decomposed vs Global Comparison ===')
comparison

## 10. Penalty-Based Soft Time Windows

Hard time windows can render some instances infeasible when traffic is heavily congested.  
A penalty-based relaxation assigns a cost to each minute of lateness:

$$\text{Objective} = \sum_{(i,j)} \tau_{ij} x_{ij} + \pi \cdot \sum_i \max(0,\, t_i - l_i)$$

where $\pi$ is the penalty weight per minute late.

In [ ]:
from src.cvrptw_solver import solve_with_penalty

print('Solving pessimistic scenario with penalty-based soft windows …')
result_penalty = solve_with_penalty(
    data,
    scenario='pessimistic',
    penalty=500,
    num_vehicles=DEFAULT_NUM_VEHICLES,
    max_weight_kg=DEFAULT_MAX_WEIGHT_KG,
    max_volume_m3=DEFAULT_MAX_VOLUME_M3,
    time_limit_s=60,
)
print(f'Status: {result_penalty.status}')
print(f'Total travel time : {result_penalty.total_travel_time:.1f} min')
print(f'Late deliveries   : {result_penalty.late_deliveries}')

# Hard constraint comparison
result_pess = scenario_results.get('pessimistic')
if result_pess and result_pess.is_solved():
    print(f'\n--- Hard TW (pessimistic) ---')
    print(f'Total travel time : {result_pess.total_travel_time:.1f} min')
    print(f'Late deliveries   : {result_pess.late_deliveries}')

## 11. Results Summary & Insights

### 11.1 Stochastic Scenario Impact

In [ ]:
print('=== Final KPI Summary ===')
kpi_df

In [ ]:
if all(s in scenario_results and scenario_results[s].is_solved() for s in ['optimistic','pessimistic']):
    opt_t   = scenario_results['optimistic'].total_travel_time
    ml_t    = scenario_results['mostlikely'].total_travel_time
    pess_t  = scenario_results['pessimistic'].total_travel_time
    uplift  = (pess_t - opt_t) / opt_t * 100

    print(f'Travel time increase (optimistic → pessimistic): +{uplift:.1f}%')
    print(f'Optimistic : {opt_t:.0f} min')
    print(f'Most-likely: {ml_t:.0f} min')
    print(f'Pessimistic: {pess_t:.0f} min')

In [ ]:
if result_ml.is_solved() and not risk_df.empty:
    high_risk_count = risk_df['high_risk'].sum()
    avg_sigma       = risk_df['sigma_arrival'].mean()
    print(f'High-risk customers (P(late)>10%): {high_risk_count}/{len(risk_df)}')
    print(f'Mean arrival σ                   : {avg_sigma:.1f} min')
    print(f'Most vulnerable customer         : Node {risk_df.iloc[0]["node"]} '
          f'(P(late)={risk_df.iloc[0]["p_late"]:.1%}, slack={risk_df.iloc[0]["slack_min"]:.0f} min)')

In [ ]:
# Experiment insights
df_fleet = exp_results['fleet_reduction']
feasible_fleet = df_fleet[df_fleet['feasible'] == 1]
if not feasible_fleet.empty:
    min_fleet = feasible_fleet['num_vehicles_available'].min()
    print(f'Minimum feasible fleet (most-likely scenario): {min_fleet} vehicles')

df_tw = exp_results['tw_tightening']
feasible_tw = df_tw[df_tw['feasible'] == 1]
if not feasible_tw.empty:
    min_lat_frac = feasible_tw['lat_fraction'].min()
    print(f'Tightest feasible LAT fraction             : {min_lat_frac:.0%} of original')

In [ ]:
if not comparison.empty:
    gap = comparison.loc['Decomposed (k-means)', 'optimality_gap_%']
    decomp_veh = comparison.loc['Decomposed (k-means)', 'vehicles_used']
    global_veh = comparison.loc['Global (monolithic)', 'vehicles_used']
    print(f'Decomposition optimality gap: {gap:+.1f}%')
    print(f'Vehicles — global: {global_veh}  vs  decomposed: {decomp_veh}')

### 11.2 Key Findings

1. **Scenario cost spread** — Pessimistic traffic inflates total travel time by a significant margin over optimistic conditions. This underscores the operational risk of planning exclusively on nominal times.

2. **Route stability** — Routes planned under most-likely conditions retain their feasibility structure in the optimistic scenario but exhibit material lateness under pessimistic traffic. The least stable routes serve customers with tight LAT = 180 min windows.

3. **High-risk nodes** — Customers positioned late in long routes accumulate variance from multiple uncertain arcs. PERT variance propagation identifies these nodes analytically, without simulation.

4. **Fleet sensitivity** — Fleet reduction below the critical threshold (identified in Experiment 3) immediately renders the instance infeasible, indicating that capacity is tight and there is no slack fleet available for disruption recovery.

5. **Time-window tightness** — Reducing LAT from its nominal value creates cascading infeasibility. The optimisation is highly sensitive to the 180-minute window customers, which are hardest to satisfy under congestion.

6. **Clustering gap** — The k-means decomposition introduces a moderate optimality gap but reduces sub-problem size dramatically. For daily operational use, this trade-off is justified when replanning speed matters.

### 11.3 Operational Recommendations

- **Priority re-routing** — High P(late) customers should be moved to the front of their routes or assigned to dedicated shorter routes.
- **Buffer time** — Add service-time buffers at high-variance arcs (identified via the arc sensitivity report).
- **Fleet dimensioning** — Maintain at least the minimum feasible fleet identified under the pessimistic scenario, not just most-likely.
- **Dynamic re-optimisation** — Consider triggered re-routing when real-time travel time exceeds PERT mean + 1.5σ on any arc.

In [ ]:
# Persist tabular results
kpi_df.to_csv(f'{OUTPUT_DIR}/scenario_kpis.csv')
exp_results['demand_scaling'].to_csv(f'{OUTPUT_DIR}/exp_demand_scaling.csv', index=False)
exp_results['tw_tightening'].to_csv(f'{OUTPUT_DIR}/exp_tw_tightening.csv', index=False)
exp_results['fleet_reduction'].to_csv(f'{OUTPUT_DIR}/exp_fleet_reduction.csv', index=False)
exp_results['capacity_sensitivity'].to_csv(f'{OUTPUT_DIR}/exp_capacity_sensitivity.csv', index=False)
if result_ml.is_solved():
    risk_df.to_csv(f'{OUTPUT_DIR}/delay_risk.csv', index=False)
    cluster_df.to_csv(f'{OUTPUT_DIR}/cluster_assignment.csv', index=False)

print('All outputs saved to outputs/')